In [2]:
# C:/Plegma_Programming/src/train_final_api_models_xgb_plegma_final.py
# ============================================================
# FINAL GENERIC UI XGBOOST API MODELS - PLEGMA
# ============================================================
#
# Trains two XGBoost models for the PLEGMA API/UI workflow:
#
#   1) with_history_generic
#      - Uses recent consumption lag / rolling features.
#      - Uses static, temporal and environmental features.
#      - DOES NOT use home_id as a model feature.
#      - DOES NOT use train-only home statistics or consumption regimes.
#      - Uses final robust XGBoost objective: reg:pseudohubererror, huber_slope=2.0.
#      - Suitable for generic UI users who provide recent consumption history.
#
#   2) no_history_simple
#      - Uses only static + temporal + environmental + appliance/sociodemographic features.
#      - DOES NOT use home_id as a model feature.
#      - DOES NOT use lag/history features.
#      - DOES NOT use behavior profiles or KNN profiles, because PLEGMA has only 10 homes.
#      - Uses final tuned squarederror XGBoost setup.
#
# Output:
#   C:/Plegma_Programming/processed/models/final_api_models/XGB/
#     ├── training_summary.json
#     ├── with_history_generic/
#     │   ├── model.joblib
#     │   ├── preprocessor.pkl
#     │   ├── feature_config.json
#     │   ├── metadata.json
#     │   ├── daily_calibrator.json
#     │   └── test_predictions.csv
#     └── no_history_simple/
#         ├── model.joblib
#         ├── preprocessor.pkl
#         ├── feature_config.json
#         ├── metadata.json
#         ├── test_predictions.csv
#         └── metrics_by_home.csv
# ============================================================

from __future__ import annotations

import json
import math
import shutil
import time
import warnings
from pathlib import Path
from typing import Any, Dict, Tuple

import joblib
import numpy as np
import pandas as pd
from xgboost import XGBRegressor

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings("ignore")

SCRIPT_VERSION = "PLEGMA_XGB_FINAL_PSEUDOHUBER_WITH_HISTORY_AND_TUNED_NO_HISTORY_V1"
print("=" * 100)
print(SCRIPT_VERSION)
print("FINAL choices: with_history_generic = pseudo-Huber slope 2.0 | no_history_simple = tuned squarederror")
print("Generic policy: home_id feature OFF | behavior profiles OFF | KNN OFF")
print("Structure aligned with final PLEGMA LGBM / RF API artifacts")
print("=" * 100)


# ============================================================
# CONFIG
# ============================================================

BASE_DIR = Path("C:/Plegma_Programming")
DATA_PATH = BASE_DIR / "PLEGMA_final_hourly_dataset.csv"

OUT_ROOT = BASE_DIR / "processed" / "models" / "final_api_models" / "XGB"
WITH_HISTORY_DIR = OUT_ROOT / "with_history_generic"
NO_HISTORY_DIR = OUT_ROOT / "no_history_simple"

TIME_COL = "timestamp"
HOME_COL = "home_id"
TARGET_COL = "consumption_Wh"

RANDOM_STATE = 42
TEST_DAYS = 30
VAL_DAYS_TOTAL = 30
CAL_DAYS = 15

TRAIN_WITH_HISTORY_GENERIC = True
TRAIN_NO_HISTORY_SIMPLE = True
CLEAR_WITH_HISTORY_DIR = True
CLEAR_NO_HISTORY_DIR = True

USE_LOG_TARGET = True
CLIP_TARGET_TRAIN_HISTORY = False
CLIP_TARGET_TRAIN_NO_HISTORY = True
CLIP_Q_LOW = 0.005
CLIP_Q_HIGH = 0.995

EARLY_STOPPING_ROUNDS = 150
UNKNOWN_LABEL = "unknown"

# PLEGMA static / environmental numeric columns.
PLEGMA_NUMERIC_BASE = [
    "hour",
    "day_of_week",
    "is_weekend",
    "is_holiday",
    "month",
    "day_of_month",
    "week_of_year",
    "hour_sin",
    "hour_cos",
    "day_sin",
    "day_cos",
    "month_sin",
    "month_cos",
    "internal_temperature",
    "external_temperature",
    "internal_humidity",
    "external_humidity",
    "num_rooms",
    "residents",
    "num_adults",
    "num_children",
    "num_elderly",
    "has_ac",
    "has_fridge_freezer",
    "has_dryer",
    "has_washing_machine",
    "has_dishwasher",
    "has_microwave",
    "has_electric_oven",
    "has_electric_hob",
    "solar_panels",
]

PLEGMA_CATEGORICAL_BASE = [
    "building_type",
    "build_era",
    "season",
    "income_band",
    "heating_type",
    "water_heater_type",
    "homeowner_status",
    "years_in_house",
    "occupation",
]

# With-history generic features. Keep max required history at 168 hours for UI practicality.
LAG_HOURS = [1, 2, 3, 6, 12, 24, 48, 72, 168]
ROLLING_WINDOWS = [3, 6, 12, 24, 48, 168]
ROLLING_EXTREME_WINDOWS = [24, 48, 168]

REQUIRED_HISTORY_COLS = [
    "lag_1h",
    "lag_24h",
    "lag_168h",
    "roll_mean_24h",
    "roll_mean_168h",
]
MIN_REQUIRED_HISTORY_HOURS = 168

# Final XGBoost params selected after PLEGMA tuning / robust search.
#
# with_history_generic:
#   Pseudo-Huber with huber_slope=2.0 improved MAE/MAPE/sMAPE vs the tuned
#   squarederror XGB while keeping RMSE and daily error practically unchanged.
#
# no_history_simple:
#   The previous tuned squarederror setup is retained. Robust/pseudo-Huber
#   no-history candidates did not improve the overall balance of RMSE/daily error.
HISTORY_XGB_PARAMS = dict(
    n_estimators=8000,
    learning_rate=0.006,
    max_depth=6,
    min_child_weight=15,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.10,
    reg_lambda=0.75,
    objective="reg:pseudohubererror",
    huber_slope=2.0,
    eval_metric="rmse",
    tree_method="hist",
    max_bin=256,
    random_state=347,
    n_jobs=-1,
    verbosity=1,
)

NO_HISTORY_XGB_PARAMS = dict(
    n_estimators=3000,
    learning_rate=0.012,
    max_depth=5,
    min_child_weight=20,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.10,
    reg_lambda=0.75,
    objective="reg:squarederror",
    eval_metric="rmse",
    tree_method="hist",
    max_bin=256,
    random_state=RANDOM_STATE + 22,
    n_jobs=-1,
    verbosity=1,
)


# ============================================================
# JSON / METRICS UTILS
# ============================================================

def to_jsonable(obj: Any) -> Any:
    if isinstance(obj, dict):
        return {str(k): to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [to_jsonable(v) for v in obj]
    if isinstance(obj, tuple):
        return [to_jsonable(v) for v in obj]
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, pd.Timestamp):
        return str(obj)
    if isinstance(obj, float) and pd.isna(obj):
        return None
    return obj


def save_json(obj: Any, path: Path) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(to_jsonable(obj), f, ensure_ascii=False, indent=2)


def file_size_mb(path: Path) -> float:
    if not path.exists():
        return float("nan")
    return path.stat().st_size / (1024 * 1024)


def mape_safe(y_true, y_pred, eps: float = 1e-6) -> float:
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    mask = y_true > eps
    if mask.sum() == 0:
        return float("nan")
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100)


def smape_safe(y_true, y_pred, eps: float = 1e-6) -> float:
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    denom = np.abs(y_true) + np.abs(y_pred) + eps
    return float(np.mean(2.0 * np.abs(y_pred - y_true) / denom) * 100)


def compute_metrics(y_true, y_pred) -> Dict[str, float]:
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(math.sqrt(mean_squared_error(y_true, y_pred))),
        "MAPE_%": mape_safe(y_true, y_pred),
        "sMAPE_%": smape_safe(y_true, y_pred),
        "bias_Wh": float(np.mean(y_pred - y_true)),
    }


def evaluate_predictions(df_eval: pd.DataFrame, pred_col: str) -> Dict[str, float]:
    y_true = df_eval[TARGET_COL].values
    y_pred = df_eval[pred_col].values
    metrics = compute_metrics(y_true, y_pred)

    daily = (
        df_eval
        .assign(date=df_eval[TIME_COL].dt.date)
        .groupby([HOME_COL, "date"], as_index=False)
        .agg(
            actual_kWh=(TARGET_COL, lambda x: x.sum() / 1000),
            pred_kWh=(pred_col, lambda x: x.sum() / 1000),
        )
    )
    daily["abs_error_kWh"] = np.abs(daily["actual_kWh"] - daily["pred_kWh"])
    daily["abs_error_pct"] = daily["abs_error_kWh"] / daily["actual_kWh"].replace(0, np.nan) * 100

    metrics.update({
        "daily_abs_error_kWh_mean": float(daily["abs_error_kWh"].mean()),
        "daily_abs_error_pct_mean": float(daily["abs_error_pct"].mean()),
        "num_home_days": int(len(daily)),
    })
    return metrics


def evaluate_by_home(df_eval: pd.DataFrame, pred_col: str) -> pd.DataFrame:
    rows = []
    for home_id, g in df_eval.groupby(HOME_COL):
        m = evaluate_predictions(g, pred_col)
        rows.append({HOME_COL: home_id, **m, "rows": int(len(g))})
    return pd.DataFrame(rows).sort_values(HOME_COL).reset_index(drop=True)


def print_metrics(title: str, metrics: Dict[str, float]) -> None:
    print("=" * 100)
    print(title)
    print("=" * 100)
    for k, v in metrics.items():
        if isinstance(v, float):
            print(f"{k}: {v:.4f}")
        else:
            print(f"{k}: {v}")


def transform_target(y):
    y = np.asarray(y, dtype=np.float64)
    if USE_LOG_TARGET:
        return np.log1p(np.maximum(y, 0.0))
    return y


def inverse_target(y_hat):
    y_hat = np.asarray(y_hat, dtype=np.float64)
    if USE_LOG_TARGET:
        return np.expm1(y_hat)
    return y_hat


def maybe_clip_target(y_train, enabled: bool):
    y_train = np.asarray(y_train, dtype=np.float64)
    if not enabled:
        return y_train, None
    lo = np.quantile(y_train, CLIP_Q_LOW)
    hi = np.quantile(y_train, CLIP_Q_HIGH)
    return np.clip(y_train, lo, hi), {"low": float(lo), "high": float(hi)}


def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)


def build_preprocessor(numeric_cols, categorical_cols):
    numeric_transformer = Pipeline(
        steps=[("imputer", SimpleImputer(strategy="median"))]
    )
    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="constant", fill_value=UNKNOWN_LABEL)),
            ("onehot", make_one_hot_encoder()),
        ]
    )
    return ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_cols),
            ("cat", categorical_transformer, categorical_cols),
        ],
        remainder="drop",
        sparse_threshold=0.3,
    )


def fit_xgb_with_early_stopping(model: XGBRegressor, X_train, y_train, X_val, y_val) -> XGBRegressor:
    """Fit XGBRegressor with compatibility across xgboost versions."""
    try:
        model.fit(
            X_train,
            y_train,
            eval_set=[(X_val, y_val)],
            verbose=200,
            early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        )
    except TypeError:
        params = model.get_params()
        params["early_stopping_rounds"] = EARLY_STOPPING_ROUNDS
        model = XGBRegressor(**params)
        model.fit(
            X_train,
            y_train,
            eval_set=[(X_val, y_val)],
            verbose=200,
        )
    return model


def get_best_n_estimators(model: XGBRegressor, fallback: int) -> int:
    best_iter = getattr(model, "best_iteration", None)
    if best_iter is None:
        try:
            best_iter = model.get_booster().best_iteration
        except Exception:
            best_iter = None
    if best_iter is None:
        return int(fallback)
    return int(best_iter) + 1


def benchmark_prediction_speed(model, X, sample_sizes=(24, 168, 720)) -> Dict[str, float]:
    results = {}
    n_available = X.shape[0]
    for n in sample_sizes:
        n_use = min(n, n_available)
        if n_use <= 0:
            continue
        X_sample = X[:n_use]
        start = time.perf_counter()
        _ = model.predict(X_sample)
        elapsed = time.perf_counter() - start
        results[f"predict_seconds_{n_use}_rows"] = float(elapsed)
    return results


# ============================================================
# COMMON FEATURE ENGINEERING
# ============================================================

def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df[TIME_COL] = pd.to_datetime(df[TIME_COL])
    df["hour"] = df[TIME_COL].dt.hour
    df["day_of_week"] = df[TIME_COL].dt.dayofweek
    df["month"] = df[TIME_COL].dt.month
    df["day_of_month"] = df[TIME_COL].dt.day
    df["week_of_year"] = df[TIME_COL].dt.isocalendar().week.astype(int)
    df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)

    if "is_holiday" not in df.columns:
        df["is_holiday"] = 0

    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
    df["day_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
    df["day_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
    return df


def temporal_split(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    max_time = df[TIME_COL].max()
    test_start = max_time - pd.Timedelta(days=TEST_DAYS) + pd.Timedelta(hours=1)
    val_start = test_start - pd.Timedelta(days=VAL_DAYS_TOTAL)
    cal_start = test_start - pd.Timedelta(days=CAL_DAYS)

    train_df = df[df[TIME_COL] < val_start].copy()
    es_df = df[(df[TIME_COL] >= val_start) & (df[TIME_COL] < cal_start)].copy()
    cal_df = df[(df[TIME_COL] >= cal_start) & (df[TIME_COL] < test_start)].copy()
    test_df = df[df[TIME_COL] >= test_start].copy()
    return train_df, es_df, cal_df, test_df


def prepare_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Ensure expected numeric columns exist.
    for col in PLEGMA_NUMERIC_BASE:
        if col not in df.columns:
            df[col] = np.nan
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Ensure expected categorical columns exist.
    for col in PLEGMA_CATEGORICAL_BASE:
        if col not in df.columns:
            df[col] = UNKNOWN_LABEL
        df[col] = df[col].fillna(UNKNOWN_LABEL).astype(str).replace({"nan": UNKNOWN_LABEL, "None": UNKNOWN_LABEL})

    return df


# ============================================================
# WITH-HISTORY GENERIC FEATURES
# ============================================================

def add_lag_and_rolling_features(df: pd.DataFrame) -> pd.DataFrame:
    """Create lag and rolling features per home.

    home_id is used only for grouping during training data preparation.
    It is not used as a model input feature.
    """
    df = df.sort_values([HOME_COL, TIME_COL]).copy()
    g = df.groupby(HOME_COL)[TARGET_COL]

    for lag in LAG_HOURS:
        df[f"lag_{lag}h"] = g.shift(lag)

    shifted = g.shift(1)

    for w in ROLLING_WINDOWS:
        df[f"roll_mean_{w}h"] = (
            shifted.groupby(df[HOME_COL])
            .rolling(w, min_periods=max(2, w // 4))
            .mean()
            .reset_index(level=0, drop=True)
        )

    for w in ROLLING_EXTREME_WINDOWS:
        df[f"roll_std_{w}h"] = (
            shifted.groupby(df[HOME_COL])
            .rolling(w, min_periods=max(2, w // 4))
            .std()
            .reset_index(level=0, drop=True)
        )
        df[f"roll_min_{w}h"] = (
            shifted.groupby(df[HOME_COL])
            .rolling(w, min_periods=max(2, w // 4))
            .min()
            .reset_index(level=0, drop=True)
        )
        df[f"roll_max_{w}h"] = (
            shifted.groupby(df[HOME_COL])
            .rolling(w, min_periods=max(2, w // 4))
            .max()
            .reset_index(level=0, drop=True)
        )

    df["lag_1h_minus_24h"] = df["lag_1h"] - df["lag_24h"]
    df["lag_24h_minus_168h"] = df["lag_24h"] - df["lag_168h"]
    df["lag_48h_minus_168h"] = df["lag_48h"] - df["lag_168h"]
    df["roll_24h_div_168h"] = df["roll_mean_24h"] / (df["roll_mean_168h"] + 1.0)
    df["roll_3h_div_24h"] = df["roll_mean_3h"] / (df["roll_mean_24h"] + 1.0)
    return df


def fit_global_daily_calibrator(cal_eval: pd.DataFrame, pred_col: str) -> Dict[str, Any]:
    y_true_sum = float(cal_eval[TARGET_COL].sum())
    y_pred_sum = float(cal_eval[pred_col].sum())
    raw_scale = (y_true_sum + 1e-6) / (y_pred_sum + 1e-6)
    raw_scale = float(np.clip(raw_scale, 0.90, 1.15))
    alpha = 0.50
    scale = float(1.0 + alpha * (raw_scale - 1.0))
    return {
        "type": "global_scale",
        "raw_scale": raw_scale,
        "alpha": alpha,
        "scale": scale,
        "default_prediction_column": "prediction_Wh",
        "optional_prediction_column": "prediction_daily_calibrated_Wh",
    }


def apply_global_daily_calibrator(pred: np.ndarray, calibrator: Dict[str, Any]) -> np.ndarray:
    pred = np.asarray(pred, dtype=np.float64)
    if calibrator.get("type") != "global_scale":
        return np.clip(pred, 0, None)
    return np.clip(pred * float(calibrator.get("scale", 1.0)), 0, None)


# ============================================================
# DATA LOADING
# ============================================================

def load_and_prepare_base_data() -> pd.DataFrame:
    print("=" * 100)
    print("Loading PLEGMA dataset")
    print("=" * 100)

    df = pd.read_csv(DATA_PATH, parse_dates=[TIME_COL], low_memory=False)
    df = df.sort_values([HOME_COL, TIME_COL]).reset_index(drop=True)

    print(f"Rows loaded: {len(df):,}")
    print(f"Homes loaded: {df[HOME_COL].nunique()}")
    print(f"Date range: {df[TIME_COL].min()} to {df[TIME_COL].max()}")

    df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
    df = df.dropna(subset=[TARGET_COL]).copy()
    df[TARGET_COL] = df[TARGET_COL].clip(lower=0)

    df = add_time_features(df)
    df = prepare_columns(df)
    return df


# ============================================================
# TRAIN WITH-HISTORY GENERIC XGB
# ============================================================

def train_with_history_generic(df_base: pd.DataFrame) -> Dict[str, Any]:
    print("=" * 100)
    print("WITH_HISTORY GENERIC TRAINING - FINAL PLEGMA XGB")
    print("=" * 100)

    if CLEAR_WITH_HISTORY_DIR and WITH_HISTORY_DIR.exists():
        print("Clearing:", WITH_HISTORY_DIR)
        shutil.rmtree(WITH_HISTORY_DIR)
    WITH_HISTORY_DIR.mkdir(parents=True, exist_ok=True)

    df_hist = add_lag_and_rolling_features(df_base)
    df_hist = df_hist.dropna(subset=REQUIRED_HISTORY_COLS).copy()
    df_hist = df_hist.sort_values([HOME_COL, TIME_COL]).reset_index(drop=True)

    print(f"Rows after history features: {len(df_hist):,}")
    print(f"Homes after history features: {df_hist[HOME_COL].nunique()}")
    print(f"Required minimum history hours: {MIN_REQUIRED_HISTORY_HOURS}")

    train_df, es_df, cal_df, test_df = temporal_split(df_hist)

    print("=" * 100)
    print("With_history generic temporal split")
    print("=" * 100)
    print(f"Train: {train_df[TIME_COL].min()} to {train_df[TIME_COL].max()} | rows: {len(train_df):,}")
    print(f"Early-stop val: {es_df[TIME_COL].min()} to {es_df[TIME_COL].max()} | rows: {len(es_df):,}")
    print(f"Calibration val: {cal_df[TIME_COL].min()} to {cal_df[TIME_COL].max()} | rows: {len(cal_df):,}")
    print(f"Test: {test_df[TIME_COL].min()} to {test_df[TIME_COL].max()} | rows: {len(test_df):,}")

    lag_cols = [c for c in df_hist.columns if c.startswith("lag_")]
    rolling_cols = [c for c in df_hist.columns if c.startswith("roll_")]

    numeric_cols = [
        c for c in PLEGMA_NUMERIC_BASE + lag_cols + rolling_cols
        if c in train_df.columns
    ]
    categorical_cols = [c for c in PLEGMA_CATEGORICAL_BASE if c in train_df.columns]

    for split_df in [train_df, es_df, cal_df, test_df]:
        for c in numeric_cols:
            split_df[c] = pd.to_numeric(split_df[c], errors="coerce")
        for c in categorical_cols:
            split_df[c] = split_df[c].fillna(UNKNOWN_LABEL).astype(str).replace({"nan": UNKNOWN_LABEL, "None": UNKNOWN_LABEL})

    print("=" * 100)
    print("With_history generic features")
    print("=" * 100)
    print(f"Numeric features: {len(numeric_cols)}")
    print(f"Categorical features: {len(categorical_cols)}")
    print("Categorical:", categorical_cols)
    print("Uses home_id as feature: False")
    print("Uses home-specific statistics: False")

    preprocessor = build_preprocessor(numeric_cols, categorical_cols)
    preprocessor.fit(train_df[numeric_cols + categorical_cols])

    X_train = preprocessor.transform(train_df[numeric_cols + categorical_cols])
    X_es = preprocessor.transform(es_df[numeric_cols + categorical_cols])
    X_cal = preprocessor.transform(cal_df[numeric_cols + categorical_cols])
    X_test = preprocessor.transform(test_df[numeric_cols + categorical_cols])

    y_train_raw = train_df[TARGET_COL].values
    y_train_used, clip_bounds = maybe_clip_target(y_train_raw, enabled=CLIP_TARGET_TRAIN_HISTORY)
    y_train = transform_target(y_train_used)
    y_es = transform_target(es_df[TARGET_COL].values)

    print("=" * 100)
    print("With_history generic matrices")
    print("=" * 100)
    print(f"X_train: {X_train.shape}")
    print(f"X_es:    {X_es.shape}")
    print(f"X_cal:   {X_cal.shape}")
    print(f"X_test:  {X_test.shape}")

    model = XGBRegressor(**HISTORY_XGB_PARAMS)
    print("=" * 100)
    print("Training WITH_HISTORY GENERIC XGB")
    print("=" * 100)
    start_train = time.perf_counter()
    model = fit_xgb_with_early_stopping(model, X_train, y_train, X_es, y_es)
    train_seconds = float(time.perf_counter() - start_train)
    best_n_estimators = get_best_n_estimators(model, HISTORY_XGB_PARAMS["n_estimators"])

    pred_cal = np.clip(inverse_target(model.predict(X_cal)), 0, None)
    pred_test = np.clip(inverse_target(model.predict(X_test)), 0, None)

    cal_eval = cal_df[[HOME_COL, TIME_COL, TARGET_COL]].copy()
    cal_eval["prediction_Wh"] = pred_cal
    daily_calibrator = fit_global_daily_calibrator(cal_eval, "prediction_Wh")
    save_json(daily_calibrator, WITH_HISTORY_DIR / "daily_calibrator.json")

    test_eval = test_df[[HOME_COL, TIME_COL, TARGET_COL]].copy()
    test_eval["prediction_Wh"] = pred_test
    test_eval["prediction_daily_calibrated_Wh"] = apply_global_daily_calibrator(pred_test, daily_calibrator)

    metrics_balanced = evaluate_predictions(test_eval, "prediction_Wh")
    metrics_daily_calibrated = evaluate_predictions(test_eval, "prediction_daily_calibrated_Wh")

    print_metrics("FINAL TEST METRICS - XGB WITH_HISTORY GENERIC BALANCED", metrics_balanced)
    print_metrics("FINAL TEST METRICS - XGB WITH_HISTORY GENERIC DAILY-CALIBRATED OPTIONAL", metrics_daily_calibrated)

    model_path = WITH_HISTORY_DIR / "model.joblib"
    preprocessor_path = WITH_HISTORY_DIR / "preprocessor.pkl"
    feature_config_path = WITH_HISTORY_DIR / "feature_config.json"
    metadata_path = WITH_HISTORY_DIR / "metadata.json"
    predictions_path = WITH_HISTORY_DIR / "test_predictions.csv"
    metrics_by_home_path = WITH_HISTORY_DIR / "metrics_by_home.csv"

    joblib.dump(model, model_path)
    joblib.dump(preprocessor, preprocessor_path)
    test_eval.to_csv(predictions_path, index=False)
    evaluate_by_home(test_eval, "prediction_Wh").to_csv(metrics_by_home_path, index=False)

    prediction_benchmark = benchmark_prediction_speed(model, X_test)

    feature_config = {
        "mode": "with_history",
        "model_family": "XGBRegressor",
        "model_name": "xgb_with_history_generic_plegma_final",
        "numeric_cols": numeric_cols,
        "categorical_cols": categorical_cols,
        "target_col": TARGET_COL,
        "time_col": TIME_COL,
        "home_col": HOME_COL,
        "use_log_target": USE_LOG_TARGET,
        "clip_target_train": CLIP_TARGET_TRAIN_HISTORY,
        "uses_home_id_as_feature": False,
        "uses_lag_features": True,
        "uses_rolling_features": True,
        "uses_home_stats": False,
        "uses_knn_profiles": False,
        "uses_behavior_profiles": False,
        "required_history_cols": REQUIRED_HISTORY_COLS,
        "lag_hours": LAG_HOURS,
        "rolling_windows": ROLLING_WINDOWS,
        "rolling_extreme_windows": ROLLING_EXTREME_WINDOWS,
        "min_required_history_hours": MIN_REQUIRED_HISTORY_HOURS,
        "default_prediction_column": "prediction_Wh",
        "optional_prediction_column": "prediction_daily_calibrated_Wh",
    }
    save_json(feature_config, feature_config_path)

    metadata = {
        "script_version": SCRIPT_VERSION,
        "model_family": "XGBRegressor",
        "model_name": "xgb_with_history_generic_plegma_final",
        "mode": "with_history",
        "xgb_params": HISTORY_XGB_PARAMS,
        "best_n_estimators": best_n_estimators,
        "use_log_target": USE_LOG_TARGET,
        "clip_target_train": CLIP_TARGET_TRAIN_HISTORY,
        "clip_bounds": clip_bounds,
        "training_seconds": train_seconds,
        "model_file_size_mb": file_size_mb(model_path),
        "preprocessor_file_size_mb": file_size_mb(preprocessor_path),
        "prediction_benchmark": prediction_benchmark,
        "metrics_balanced": metrics_balanced,
        "metrics_daily_calibrated_optional": metrics_daily_calibrated,
        "train_period": {"start": str(train_df[TIME_COL].min()), "end": str(train_df[TIME_COL].max()), "rows": int(len(train_df))},
        "early_stop_validation_period": {"start": str(es_df[TIME_COL].min()), "end": str(es_df[TIME_COL].max()), "rows": int(len(es_df))},
        "calibration_validation_period": {"start": str(cal_df[TIME_COL].min()), "end": str(cal_df[TIME_COL].max()), "rows": int(len(cal_df))},
        "test_period": {"start": str(test_df[TIME_COL].min()), "end": str(test_df[TIME_COL].max()), "rows": int(len(test_df))},
        "artifact_files": {
            "model": str(model_path),
            "preprocessor": str(preprocessor_path),
            "feature_config": str(feature_config_path),
            "metadata": str(metadata_path),
            "daily_calibrator": str(WITH_HISTORY_DIR / "daily_calibrator.json"),
            "predictions": str(predictions_path),
            "metrics_by_home": str(metrics_by_home_path),
        },
    }
    save_json(metadata, metadata_path)

    return {
        "metrics_balanced": metrics_balanced,
        "metrics_daily_calibrated_optional": metrics_daily_calibrated,
        "model_path": str(model_path),
        "preprocessor_path": str(preprocessor_path),
        "feature_config_path": str(feature_config_path),
        "metadata_path": str(metadata_path),
        "predictions_path": str(predictions_path),
        "best_n_estimators": best_n_estimators,
        "training_seconds": train_seconds,
        "model_file_size_mb": file_size_mb(model_path),
    }


# ============================================================
# TRAIN NO-HISTORY SIMPLE XGB
# ============================================================

def train_no_history_simple(df_base: pd.DataFrame) -> Dict[str, Any]:
    print("=" * 100)
    print("NO_HISTORY SIMPLE TRAINING - FINAL PLEGMA XGB")
    print("=" * 100)

    if CLEAR_NO_HISTORY_DIR and NO_HISTORY_DIR.exists():
        print("Clearing:", NO_HISTORY_DIR)
        shutil.rmtree(NO_HISTORY_DIR)
    NO_HISTORY_DIR.mkdir(parents=True, exist_ok=True)

    df_no = df_base.sort_values(TIME_COL).reset_index(drop=True).copy()
    train_df, es_df, cal_df, test_df = temporal_split(df_no)

    final_train_df = (
        pd.concat([train_df, es_df, cal_df], axis=0)
        .sort_values(TIME_COL)
        .reset_index(drop=True)
    )

    print("=" * 100)
    print("No_history simple temporal split")
    print("=" * 100)
    print(f"Train: {train_df[TIME_COL].min()} to {train_df[TIME_COL].max()} | rows: {len(train_df):,}")
    print(f"Early-stop val: {es_df[TIME_COL].min()} to {es_df[TIME_COL].max()} | rows: {len(es_df):,}")
    print(f"Calibration val: {cal_df[TIME_COL].min()} to {cal_df[TIME_COL].max()} | rows: {len(cal_df):,}")
    print(f"Final train used: {final_train_df[TIME_COL].min()} to {final_train_df[TIME_COL].max()} | rows: {len(final_train_df):,}")
    print(f"Test: {test_df[TIME_COL].min()} to {test_df[TIME_COL].max()} | rows: {len(test_df):,}")

    numeric_cols = [c for c in PLEGMA_NUMERIC_BASE if c in final_train_df.columns]
    categorical_cols = [c for c in PLEGMA_CATEGORICAL_BASE if c in final_train_df.columns]

    for split_df in [train_df, es_df, cal_df, final_train_df, test_df]:
        for c in numeric_cols:
            split_df[c] = pd.to_numeric(split_df[c], errors="coerce")
        for c in categorical_cols:
            split_df[c] = split_df[c].fillna(UNKNOWN_LABEL).astype(str).replace({"nan": UNKNOWN_LABEL, "None": UNKNOWN_LABEL})

    print("=" * 100)
    print("No_history simple features")
    print("=" * 100)
    print(f"Numeric features: {len(numeric_cols)}")
    print(f"Categorical features: {len(categorical_cols)}")
    print("Numeric:", numeric_cols)
    print("Categorical:", categorical_cols)
    print("Uses home_id as feature: False")
    print("Uses lag/history features: False")
    print("Uses behavior profiles: False")
    print("Uses KNN profiles: False")

    # Tuning phase: train only on train_df, validate on early-stop period.
    preprocessor_tune = build_preprocessor(numeric_cols, categorical_cols)
    preprocessor_tune.fit(train_df[numeric_cols + categorical_cols])

    X_train_tune = preprocessor_tune.transform(train_df[numeric_cols + categorical_cols])
    X_es_tune = preprocessor_tune.transform(es_df[numeric_cols + categorical_cols])

    y_train_raw = train_df[TARGET_COL].values
    y_train_used, tune_clip_bounds = maybe_clip_target(y_train_raw, enabled=CLIP_TARGET_TRAIN_NO_HISTORY)
    y_train_tune = transform_target(y_train_used)
    y_es_tune = transform_target(es_df[TARGET_COL].values)

    print("=" * 100)
    print("No_history simple tuning matrices")
    print("=" * 100)
    print(f"X_train_tune: {X_train_tune.shape}")
    print(f"X_es_tune:    {X_es_tune.shape}")

    tune_model = XGBRegressor(**NO_HISTORY_XGB_PARAMS)
    print("=" * 100)
    print("Tuning NO_HISTORY SIMPLE XGB")
    print("=" * 100)
    tune_model = fit_xgb_with_early_stopping(tune_model, X_train_tune, y_train_tune, X_es_tune, y_es_tune)
    best_n_estimators = get_best_n_estimators(tune_model, NO_HISTORY_XGB_PARAMS["n_estimators"])

    # Final refit: fit on train + early-stop val + calibration val before test.
    final_params = dict(NO_HISTORY_XGB_PARAMS)
    final_params["n_estimators"] = best_n_estimators
    final_params["random_state"] = RANDOM_STATE + 101

    preprocessor = build_preprocessor(numeric_cols, categorical_cols)
    preprocessor.fit(final_train_df[numeric_cols + categorical_cols])

    X_final_train = preprocessor.transform(final_train_df[numeric_cols + categorical_cols])
    X_test = preprocessor.transform(test_df[numeric_cols + categorical_cols])

    y_final_raw = final_train_df[TARGET_COL].values
    y_final_used, final_clip_bounds = maybe_clip_target(y_final_raw, enabled=CLIP_TARGET_TRAIN_NO_HISTORY)
    y_final = transform_target(y_final_used)

    print("=" * 100)
    print("No_history simple final matrices")
    print("=" * 100)
    print(f"X_final_train: {X_final_train.shape}")
    print(f"X_test:        {X_test.shape}")
    print(f"Selected n_estimators from early stopping: {best_n_estimators}")

    model = XGBRegressor(**final_params)
    print("=" * 100)
    print("Final refit NO_HISTORY SIMPLE XGB")
    print("=" * 100)
    start_train = time.perf_counter()
    model.fit(X_final_train, y_final, verbose=False)
    train_seconds = float(time.perf_counter() - start_train)

    pred_test = np.clip(inverse_target(model.predict(X_test)), 0, None)

    test_eval = test_df[[HOME_COL, TIME_COL, TARGET_COL]].copy()
    test_eval["prediction_Wh"] = pred_test

    metrics = evaluate_predictions(test_eval, "prediction_Wh")
    print_metrics("FINAL TEST METRICS - XGB NO_HISTORY SIMPLE", metrics)

    model_path = NO_HISTORY_DIR / "model.joblib"
    preprocessor_path = NO_HISTORY_DIR / "preprocessor.pkl"
    feature_config_path = NO_HISTORY_DIR / "feature_config.json"
    metadata_path = NO_HISTORY_DIR / "metadata.json"
    predictions_path = NO_HISTORY_DIR / "test_predictions.csv"
    metrics_by_home_path = NO_HISTORY_DIR / "metrics_by_home.csv"

    joblib.dump(model, model_path)
    joblib.dump(preprocessor, preprocessor_path)
    test_eval.to_csv(predictions_path, index=False)
    evaluate_by_home(test_eval, "prediction_Wh").to_csv(metrics_by_home_path, index=False)

    prediction_benchmark = benchmark_prediction_speed(model, X_test)

    feature_config = {
        "mode": "no_history",
        "model_family": "XGBRegressor",
        "model_name": "xgb_no_history_simple_plegma_final",
        "numeric_cols": numeric_cols,
        "categorical_cols": categorical_cols,
        "target_col": TARGET_COL,
        "time_col": TIME_COL,
        "home_col": HOME_COL,
        "use_log_target": USE_LOG_TARGET,
        "clip_target_train": CLIP_TARGET_TRAIN_NO_HISTORY,
        "uses_home_id_as_feature": False,
        "uses_lag_features": False,
        "uses_rolling_features": False,
        "uses_home_stats": False,
        "uses_knn_profiles": False,
        "uses_behavior_profiles": False,
        "default_prediction_column": "prediction_Wh",
    }
    save_json(feature_config, feature_config_path)

    metadata = {
        "script_version": SCRIPT_VERSION,
        "model_family": "XGBRegressor",
        "model_name": "xgb_no_history_simple_plegma_final",
        "mode": "no_history",
        "xgb_params_tuning": NO_HISTORY_XGB_PARAMS,
        "xgb_params_final": final_params,
        "best_n_estimators": best_n_estimators,
        "use_log_target": USE_LOG_TARGET,
        "clip_target_train": CLIP_TARGET_TRAIN_NO_HISTORY,
        "tune_clip_bounds": tune_clip_bounds,
        "final_clip_bounds": final_clip_bounds,
        "training_seconds": train_seconds,
        "model_file_size_mb": file_size_mb(model_path),
        "preprocessor_file_size_mb": file_size_mb(preprocessor_path),
        "prediction_benchmark": prediction_benchmark,
        "metrics": metrics,
        "train_period": {"start": str(train_df[TIME_COL].min()), "end": str(train_df[TIME_COL].max()), "rows": int(len(train_df))},
        "early_stop_validation_period": {"start": str(es_df[TIME_COL].min()), "end": str(es_df[TIME_COL].max()), "rows": int(len(es_df))},
        "calibration_validation_period_used_in_final_refit": {"start": str(cal_df[TIME_COL].min()), "end": str(cal_df[TIME_COL].max()), "rows": int(len(cal_df))},
        "final_train_period": {"start": str(final_train_df[TIME_COL].min()), "end": str(final_train_df[TIME_COL].max()), "rows": int(len(final_train_df))},
        "test_period": {"start": str(test_df[TIME_COL].min()), "end": str(test_df[TIME_COL].max()), "rows": int(len(test_df))},
        "artifact_files": {
            "model": str(model_path),
            "preprocessor": str(preprocessor_path),
            "feature_config": str(feature_config_path),
            "metadata": str(metadata_path),
            "predictions": str(predictions_path),
            "metrics_by_home": str(metrics_by_home_path),
        },
    }
    save_json(metadata, metadata_path)

    return {
        "metrics": metrics,
        "model_path": str(model_path),
        "preprocessor_path": str(preprocessor_path),
        "feature_config_path": str(feature_config_path),
        "metadata_path": str(metadata_path),
        "predictions_path": str(predictions_path),
        "best_n_estimators": best_n_estimators,
        "training_seconds": train_seconds,
        "model_file_size_mb": file_size_mb(model_path),
    }


# ============================================================
# MAIN
# ============================================================

def main() -> None:
    OUT_ROOT.mkdir(parents=True, exist_ok=True)

    df_base = load_and_prepare_base_data()

    summary = {
        "script_version": SCRIPT_VERSION,
        "data_path": str(DATA_PATH),
        "out_root": str(OUT_ROOT),
        "target_col": TARGET_COL,
        "time_col": TIME_COL,
        "home_col": HOME_COL,
        "test_days": TEST_DAYS,
        "val_days_total": VAL_DAYS_TOTAL,
        "cal_days": CAL_DAYS,
        "use_log_target": USE_LOG_TARGET,
        "policy": {
            "generic_ui": True,
            "home_id_as_feature": False,
            "behavior_profiles": False,
            "knn_profiles": False,
        },
        "models": {},
    }

    if TRAIN_WITH_HISTORY_GENERIC:
        summary["models"]["with_history_generic"] = train_with_history_generic(df_base)

    if TRAIN_NO_HISTORY_SIMPLE:
        summary["models"]["no_history_simple"] = train_no_history_simple(df_base)

    summary_path = OUT_ROOT / "training_summary.json"
    save_json(summary, summary_path)

    print("=" * 100)
    print("FINAL PLEGMA GENERIC XGB API MODELS TRAINING COMPLETED")
    print("=" * 100)
    print("OUT_ROOT:", OUT_ROOT)
    print("With_history generic folder:", WITH_HISTORY_DIR)
    print("No_history simple folder:", NO_HISTORY_DIR)
    print("Training summary:", summary_path)
    if "with_history_generic" in summary["models"]:
        print("With_history generic model:", summary["models"]["with_history_generic"]["model_path"])
    if "no_history_simple" in summary["models"]:
        print("No_history simple model:", summary["models"]["no_history_simple"]["model_path"])


if __name__ == "__main__":
    main()


PLEGMA_XGB_FINAL_PSEUDOHUBER_WITH_HISTORY_AND_TUNED_NO_HISTORY_V1
FINAL choices: with_history_generic = pseudo-Huber slope 2.0 | no_history_simple = tuned squarederror
Generic policy: home_id feature OFF | behavior profiles OFF | KNN OFF
Structure aligned with final PLEGMA LGBM / RF API artifacts
Loading PLEGMA dataset
Rows loaded: 71,111
Homes loaded: 10
Date range: 2022-07-11 14:00:00 to 2023-10-01 01:00:00
WITH_HISTORY GENERIC TRAINING - FINAL PLEGMA XGB
Clearing: C:\Plegma_Programming\processed\models\final_api_models\XGB\with_history_generic
Rows after history features: 69,431
Homes after history features: 10
Required minimum history hours: 168
With_history generic temporal split
Train: 2022-07-18 14:00:00 to 2023-08-02 01:00:00 | rows: 56,475
Early-stop val: 2023-08-02 02:00:00 to 2023-08-17 01:00:00 | rows: 3,240
Calibration val: 2023-08-17 02:00:00 to 2023-09-01 01:00:00 | rows: 3,240
Test: 2023-09-01 02:00:00 to 2023-10-01 01:00:00 | rows: 6,476
With_history generic features
N